# ДЗ 2. Dataset, DataLoader и корректная работа с данными

Задание к теме «Dataset/DataLoader, train/val/test split». На лекции вы видели `FashionMNIST`,
`ImageFolder`, `random_split`, `transforms.Compose` и базовый `DataLoader`. Здесь мы пойдём
дальше: `Dataset`/`DataLoader` — это общий механизм PyTorch для *любых* данных, не только
картинок, и большая часть реальных багов в ML-пайплайнах прячется именно на этом этапе — не в
архитектуре модели.

Все задачи используют синтетические или встроенные в `sklearn` данные (не требуют скачивания
и работают одинаково что в Colab, что офлайн), считаются секунды — минуты на CPU.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
torch.manual_seed(0)
np.random.seed(0)


## Уровень 1. Попроще

### Задача 1.1. Свой `Dataset` для табличных данных (с нормализацией и аугментацией)

Оберните любой табличный датасет (например, `sklearn.datasets.load_wine` — он не требует
скачивания) в собственный класс `NormalizingTabularDataset(Dataset)`:

1. В конструкторе примите сырые признаки `X` (numpy-массив) и метки `y`, посчитайте по всей
   выборке `X` среднее и стандартное отклонение каждого признака (z-score нормализация) и
   сохраните их как атрибуты объекта.
2. В `__getitem__` возвращайте нормализованный объект `(x - mean) / std` (в виде тензора) и
   соответствующую метку.
3. Добавьте параметры конструктора `train: bool = False` и `noise_std: float = 0.0`. Если
   `train=True` и `noise_std > 0`, добавляйте к нормализованному объекту гауссовский шум с этим
   стандартным отклонением (простая аугментация для табличных данных) — новый шум при каждом
   обращении к `__getitem__`. Если `train=False`, объект должен быть строго детерминирован
   (шум не добавляется независимо от значения `noise_std`).

Убедитесь, что `DataLoader` поверх вашего класса отдаёт батчи ожидаемой формы, и продемонстрируйте,
что при `train=True` повторные обращения к одному и тому же индексу дают разные тензоры, а при
`train=False` — совпадающие.


In [2]:
from sklearn.datasets import load_wine
import pandas as pd


class NormalizingTabularDataset(Dataset):
    def __init__(self, X, y, train=False, noise_std=0.0, stats=None):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y)
        self.mean, self.std = stats if stats is not None else (self.X.mean(0), self.X.std(0))
        self.train = train
        self.noise_std = noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = (self.X[i] - self.mean) / self.std
        if self.train and self.noise_std > 0:
            x = x + self.noise_std * torch.randn_like(x)
        return x, self.y[i]


X, y = load_wine(return_X_y=True)
perm = np.random.permutation(len(y))
tr, va = perm[:140], perm[140:]

train_ds = NormalizingTabularDataset(X[tr], y[tr], train=True, noise_std=0.1)
val_ds = NormalizingTabularDataset(X[va], y[va], noise_std=0.1, stats=(train_ds.mean, train_ds.std))

xb, yb = next(iter(DataLoader(train_ds, batch_size=32, shuffle=True)))
assert xb.shape == (32, 13) and yb.shape == (32,)
assert not torch.equal(train_ds[0][0], train_ds[0][0])
assert torch.equal(val_ds[0][0], val_ds[0][0])

clean = NormalizingTabularDataset(X[tr], y[tr])
Z, _ = next(iter(DataLoader(clean, batch_size=len(clean))))
assert Z.mean(0).abs().max() < 1e-5 and (Z.std(0) - 1).abs().max() < 1e-5

noise = torch.stack([train_ds[0][0] - clean[0][0] for _ in range(2000)])
noise.std().item()

0.10065527260303497

Параметр `stats` добавлен сверх условия: если статистики считать отдельно на валидации, в её нормализацию утекает информация о самой валидации, а модель на train и val видит признаки в чуть разных шкалах. Поэтому val-датасет получает mean/std от train-датасета, а `noise_std=0.1` у него передан специально, чтобы показать, что при `train=False` шум игнорируется

Последняя строка проверяет саму аугментацию: разность зашумлённого и чистого объекта имеет std ≈ 0.1, то есть шум добавляется ровно заданной величины и уже после нормализации

### Задача 1.2. Стратифицированный train/test split без искажения баланса классов

Представьте, что среди объектов один класс сильно **редкий** (например, при бинарной
классификации редких событий: мошеннические транзакции, редкие заболевания и т.п.). Если
разбивать выборку на train/test **совершенно случайно** (без учёта меток), при небольшом размере
теста доля редкого класса в train и test может случайно сильно разойтись с исходной — вплоть до
того, что в test не попадёт вообще ни одного объекта редкого класса, и оценить качество модели на
нём станет невозможно.

1. Сгенерируйте синтетические метки: 1000 объектов, 3 класса с сильно разными частотами
   (например, 800/150/50).
2. Реализуйте `stratified_split(labels, test_frac)`, который возвращает индексы train и test так,
   чтобы **доля каждого класса** в train и в test была одинаковой (с точностью до округления) и
   равнялась доле этого класса во всей выборке — то есть сплит делается независимо **внутри
   каждого класса**, а не по всей выборке целиком.
3. Продемонстрируйте на этом же массиве `labels`, что наивный `sklearn.model_selection.
   train_test_split` без параметра `stratify` (или ручной случайный сплит по всем объектам сразу)
   может дать заметно смещённые пропорции классов в test, особенно для редкого класса: запустите
   наивный сплит несколько раз с разными `random_state` и посчитайте, насколько сильно доля
   редкого класса в test отклоняется от его доли в полной выборке — в среднем и в худшем случае.


In [3]:
from sklearn.model_selection import train_test_split

labels = np.random.permutation(np.repeat([0, 1, 2], [800, 150, 50]))


def stratified_split(labels, test_frac, seed=0):
    rng = np.random.default_rng(seed)
    _, cls, counts = np.unique(labels, return_inverse=True, return_counts=True)
    order = rng.permutation(len(labels))
    order = order[np.argsort(cls[order], kind='stable')]
    rank = np.arange(len(labels)) - np.repeat(np.cumsum(counts) - counts, counts)
    is_test = rank < np.repeat(np.round(counts * test_frac).astype(int), counts)
    return order[~is_test], order[is_test]


def shares(idx):
    return np.bincount(labels[idx], minlength=3) / len(idx)


tr, te = stratified_split(labels, 0.1)
assert len(np.intersect1d(tr, te)) == 0 and len(tr) + len(te) == len(labels)
assert np.allclose(shares(tr), shares(np.arange(len(labels))), atol=1e-3)
assert np.allclose(shares(te), shares(np.arange(len(labels))), atol=1e-3)

pd.DataFrame([shares(np.arange(len(labels))), shares(tr), shares(te)],
             index=['вся выборка', 'train', 'test'], columns=['класс 0', 'класс 1', 'класс 2'])

,класс 0,класс 1,класс 2
вся выборка,0.8,0.15,0.05
train,0.8,0.15,0.05
test,0.8,0.15,0.05


In [4]:
p, n, m = (labels == 2).mean(), len(labels), 100
rare_in_test = np.array([
    (labels[train_test_split(np.arange(n), test_size=m, random_state=s)[1]] == 2).mean()
    for s in range(2000)
])
dev = np.abs(rare_in_test - p)

pd.Series({
    'доля класса 2 во всей выборке': p,
    'среднее |отклонение| в test': dev.mean(),
    'худшее отклонение': dev.max(),
    'std по гипергеометрическому закону': np.sqrt(p * (1 - p) / m * (n - m) / (n - 1)),
    'доля сплитов без класса 2 в test': (rare_in_test == 0).mean(),
    'доля сплитов, где класс 2 ушёл в 2 раза и больше': ((rare_in_test <= p / 2) | (rare_in_test >= 2 * p)).mean(),
}).round(4)

доля класса 2 во всей выборке                       0.0500
среднее |отклонение| в test                         0.0164
худшее отклонение                                   0.0900
std по гипергеометрическому закону                  0.0207
доля сплитов без класса 2 в test                    0.0050
доля сплитов, где класс 2 ушёл в 2 раза и больше    0.1270
dtype: float64

Сплит векторный, без цикла по классам: объекты случайно перемешиваются, затем стабильно сортируются по классу, так что внутри каждого класса порядок остаётся случайным. Номер объекта внутри своего класса (`rank`) получается вычитанием начала блока класса, и в test уходят первые round(frac·count) объектов каждого блока

Наивный сплит на 100 объектов в test ошибается в доле редкого класса в среднем примерно на треть от неё самой. Это не случайность реализации: число редких объектов в случайном test подчиняется гипергеометрическому распределению, и эмпирический разброс совпадает с его теоретическим std. Примерно каждый восьмой сплит даёт долю класса 2 в test вдвое меньше или больше нужной, изредка класса 2 в test нет вообще. На таком test метрики по редкому классу либо не считаются, либо шумят сильнее, чем различаются модели

### Задача 1.3. `WeightedRandomSampler` для многоклассового дисбаланса

Постройте синтетический **трёхклассовый** датасет с сильным дисбалансом (700 объектов класса 0,
200 — класса 1, 100 — класса 2). Реализуйте функцию `make_balancing_weights(labels)`, которая для
тензора или массива меток **произвольного** числа классов возвращает тензор весов для каждого
объекта, обратно пропорциональных частоте его класса (без хардкода числа классов или конкретных
частот — функция должна одинаково работать и на 2, и на 5 классах). Сравните распределение классов
в батчах обычного `DataLoader(shuffle=True)` и `DataLoader` с
`sampler=WeightedRandomSampler(weights=make_balancing_weights(labels), ...)`. Посчитайте среднюю
долю **каждого** класса по нескольким батчам в обоих случаях и убедитесь, что с весами все три
класса представлены в батчах примерно поровну.


In [5]:
from torch.utils.data import TensorDataset


def make_balancing_weights(labels):
    labels = torch.as_tensor(labels)
    _, inv, counts = torch.unique(labels, return_inverse=True, return_counts=True)
    return (1.0 / counts.double())[inv]


for sizes in ([30, 970], [5, 50, 500, 17, 1]):
    lab = torch.repeat_interleave(torch.tensor([3, -1, 7, 0, 42][:len(sizes)]), torch.tensor(sizes))
    w = make_balancing_weights(lab)
    per_class = torch.zeros(len(sizes), dtype=w.dtype).index_add_(0, torch.unique(lab, return_inverse=True)[1], w)
    assert torch.allclose(per_class, torch.ones(len(sizes), dtype=w.dtype))

y3 = torch.repeat_interleave(torch.arange(3), torch.tensor([700, 200, 100]))
ds3 = TensorDataset(torch.randn(len(y3), 5) + y3[:, None], y3)
sampler = WeightedRandomSampler(make_balancing_weights(y3), num_samples=len(y3), replacement=True)


def batch_shares(loader):
    return torch.stack([torch.bincount(yb, minlength=3) / len(yb) for _, yb in loader]).mean(0)


plain = batch_shares(DataLoader(ds3, batch_size=64, shuffle=True, drop_last=True))
balanced = batch_shares(DataLoader(ds3, batch_size=64, sampler=sampler, drop_last=True))
assert (balanced - 1 / 3).abs().max() < 0.06

seen = torch.zeros(len(y3), dtype=torch.bool)
seen[list(sampler)] = True
coverage = torch.stack([seen[y3 == c].float().mean() for c in range(3)])

pd.DataFrame(torch.stack([plain, balanced, coverage]).numpy(),
             index=['доли в батче, shuffle=True', 'доли в батче, сэмплер', 'покрытие класса за эпоху, сэмплер'],
             columns=['класс 0', 'класс 1', 'класс 2']).round(3)

,класс 0,класс 1,класс 2
"доли в батче, shuffle=True",0.699,0.20,0.101
"доли в батче, сэмплер",0.343,0.32,0.338
"покрытие класса за эпоху, сэмплер",0.373,0.86,0.950


Веса считаются через `torch.unique(return_inverse=True)`, поэтому метки не обязаны быть 0..K−1: в проверке используются метки вида −1, 42 и т.п. Суммарный вес каждого класса равен 1, это и есть условие равной вероятности классов при сэмплировании

Последняя строка таблицы показывает цену балансировки с `replacement=True` при `num_samples=len(dataset)`: каждый объект класса 2 за эпоху выпадает в среднем больше трёх раз, а больше 60% объектов класса 0 за эпоху не попадают в батчи ни разу. Мажоритарный класс фактически недоиспользуется, поэтому для такого сэмплера имеет смысл либо увеличивать `num_samples`, либо считать прогресс обучения в шагах, а не в эпохах

## Уровень 2. Посложнее

### Задача 2. Кастомный `collate_fn` для пар параллельных последовательностей переменной длины

Во многих реальных задачах объект — это не одна последовательность, а **несколько параллельных**
последовательностей одинаковой (для данного объекта) длины: например, последовательность токенов
текста и последовательность длительностей пауз между ними, или последовательность ID событий и
последовательность времени с предыдущего события. При паддинге такого объекта важно паддить обе
последовательности **одной и той же** длиной и использовать **одну и ту же** маску.

1. Реализуйте `PairSeqDataset`, возвращающий тройки: (целочисленная последовательность случайной
   длины $L$ — «токены», вещественная последовательность той же длины $L$ — «временные
   интервалы», бинарная метка).
2. Реализуйте `pad_pair_collate(batch)` — функцию для параметра `collate_fn` в `DataLoader`,
   которая паддит обе последовательности нулями до длины максимальной именно в этом батче и
   возвращает: паддинговый тензор токенов, паддинговый тензор временных интервалов, общую маску
   реальных (не паддинговых) позиций, исходные длины и метки.
3. Убедитесь, что: (а) число реальных позиций, посчитанное по маске, совпадает с суммой исходных
   длин последовательностей; (б) маска, применённая к паддинговому тензору токенов и к
   паддинговому тензору интервалов, действительно обнуляет одни и те же позиции в обоих (то есть
   паддинг синхронизирован между двумя последовательностями).


In [6]:
class PairSeqDataset(Dataset):
    def __init__(self, n, vocab_size=50, min_len=3, max_len=20, seed=0):
        self.n, self.vocab_size, self.seed = n, vocab_size, seed
        self.min_len, self.max_len = min_len, max_len

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        g = torch.Generator().manual_seed(self.seed * 1_000_003 + i)
        L = int(torch.randint(self.min_len, self.max_len + 1, (1,), generator=g))
        tokens = torch.randint(1, self.vocab_size, (L,), generator=g)
        gaps = torch.empty(L).exponential_(generator=g)
        return tokens, gaps, int(gaps.mean() > 1)


def pad_pair_collate(batch):
    tokens, gaps, labels = zip(*batch)
    lengths = torch.tensor([len(t) for t in tokens])
    if not torch.equal(lengths, torch.tensor([len(g) for g in gaps])):
        raise ValueError('длины tokens и gaps внутри объекта не совпадают')
    mask = torch.arange(int(lengths.max())) < lengths[:, None]
    tok_pad = torch.zeros(mask.shape, dtype=tokens[0].dtype)
    gap_pad = torch.zeros(mask.shape, dtype=gaps[0].dtype)
    tok_pad[mask] = torch.cat(tokens)
    gap_pad[mask] = torch.cat(gaps)
    return tok_pad, gap_pad, mask, lengths, torch.tensor(labels)


ds = PairSeqDataset(500)
assert all(torch.equal(a, b) for a, b in zip(ds[7][:2], ds[7][:2]))

bs = 16
for k, (tok, gap, mask, lengths, y) in enumerate(DataLoader(ds, batch_size=bs, collate_fn=pad_pair_collate)):
    assert tok.shape == gap.shape == mask.shape == (len(lengths), lengths.max())
    assert mask.sum() == lengths.sum()
    assert torch.equal(tok * mask, tok) and torch.equal(gap * mask, gap)
    assert torch.equal(tok != 0, mask) and torch.equal(gap != 0, mask)
    parts = zip(tok[mask].split(lengths.tolist()), gap[mask].split(lengths.tolist()))
    for j, (t, g) in enumerate(parts):
        t0, g0, _ = ds[k * bs + j]
        assert torch.equal(t, t0) and torch.equal(g, g0)

tok, gap, mask, lengths, y = next(iter(DataLoader(ds, batch_size=4, shuffle=True, collate_fn=pad_pair_collate)))
tok, mask.int(), lengths

(tensor([[31, 12, 17, 43, 36,  7, 11,  9, 18,  0,  0,  0,  0,  0,  0,  0],
         [48, 29, 37, 23,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [41, 31, 23,  3,  3, 38,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [19, 30, 20, 47, 44, 49, 47, 26, 30, 18,  7,  3, 32, 26, 34, 19]]),
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
         [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32),
 tensor([ 9,  4,  6, 16]))

Паддинг сделан без `pad_sequence`: маска строится один раз из длин (`arange(T) < lengths[:, None]`), после чего обе последовательности батча склеиваются и записываются в нули по этой маске. Булева индексация проходит позиции построчно, в том же порядке, в каком идут склеенные последовательности, поэтому синхронность паддинга гарантирована самим способом записи, а не проверкой после

Токены генерируются из диапазона 1..vocab−1, интервалы экспоненциальные и строго положительны, так что 0 в обоих тензорах встречается только на паддинге. Это позволяет проверить синхронизацию в самой сильной форме: множество ненулевых позиций токенов, множество ненулевых позиций интервалов и маска совпадают поэлементно. Дополнительно по маске и длинам восстанавливаются исходные последовательности и сравниваются с тем, что отдаёт датасет

Датасет ничего не хранит: объект с индексом i генерируется из собственного `Generator` с сидом, зависящим от i, поэтому одинаков при любом числе обращений, порядке чтения и числе воркеров DataLoader

---
**Что сдавать:** ноутбук с реализованными классами/функциями, пройденными проверками и
письменными ответами на вопросы для решённых задач. Всё задание в сумме считается на CPU не
дольше нескольких минут.
